In [2]:
import os
import json
from datetime import date, datetime, timezone
 
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import psycopg2
from psycopg2.extras import Json
import pandas as pd
from dotenv import load_dotenv
 
from Coin_mapping import COIN_TO_BINANCE
 
LOG_FILE = "daily_pipeline_log.txt"

In [3]:
def log(message):
    timestamp = datetime.now(timezone.utc).isoformat()
    line = f"[{timestamp}] {message}"
    print(line)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

In [4]:
def make_session():
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=2, status_forcelist=[429, 451, 500, 502, 503])
    session.mount("https://", HTTPAdapter(max_retries=retries))
    return session

In [5]:
def fetch_coingecko(session, coingecko_api_key, coin_ids):
    ids_param = ",".join(coin_ids)
    url = (
        "https://api.coingecko.com/api/v3/simple/price"
        f"?ids={ids_param}&vs_currencies=usd"
        "&include_market_cap=true&include_24hr_vol=true"
        f"&x_cg_demo_api_key={coingecko_api_key}"
    )
    try:
        response = session.get(url, timeout=15)
    except requests.exceptions.RequestException as e:
        log(f"NETWORK ERROR calling CoinGecko: {e}")
        return None, None
 
    if response.status_code != 200:
        log(f"CoinGecko returned HTTP {response.status_code}: {response.text[:200]}")
        return response.status_code, None
 
    return response.status_code, response.json()

In [6]:
def fetch_binance(session, binance_symbols):
    url = "https://api.binance.com/api/v3/ticker/24hr"
    params = {"symbols": json.dumps(binance_symbols, separators=(",", ":"))}
    try:
        response = session.get(url, params=params, timeout=15)
    except requests.exceptions.RequestException as e:
        log(f"NETWORK ERROR calling Binance: {e}")
        return None, None
 
    if response.status_code != 200:
        log(f"Binance returned HTTP {response.status_code}: {response.text[:200]}")
        return response.status_code, None
 
    return response.status_code, response.json()

In [7]:
def save_raw(conn, today, cg_status, cg_payload, bn_status, bn_payload):
    cursor = conn.cursor()
    upsert_query = """
        INSERT INTO raw_ingestion_log (pull_date, source, http_status, payload, ingested_at)
        VALUES (%s, %s, %s, %s, CURRENT_TIMESTAMP)
        ON CONFLICT (pull_date, source)
        DO UPDATE SET
            payload = EXCLUDED.payload,
            http_status = EXCLUDED.http_status,
            ingested_at = CURRENT_TIMESTAMP;
    """
    if cg_payload is not None:
        cursor.execute(upsert_query, (today, "coingecko", cg_status, Json(cg_payload)))
    if bn_payload is not None:
        cursor.execute(upsert_query, (today, "binance", bn_status, Json(bn_payload)))
    conn.commit()
    cursor.close()
    log("Raw payloads saved to raw_ingestion_log.")

In [8]:

def build_metrics_rows(today, cg_payload, bn_payload):
    binance_by_symbol = {t["symbol"]: t for t in (bn_payload or [])}
 
    rows = []
    for coin_id, cg_values in (cg_payload or {}).items():
        binance_symbol = COIN_TO_BINANCE.get(coin_id)
        ticker = binance_by_symbol.get(binance_symbol) if binance_symbol else None
 
        if ticker is None:
            log(f"No Binance data for {coin_id} ({binance_symbol}) today - skipping this coin.")
            continue
 
        cg_price = cg_values.get("usd")
        binance_price = float(ticker["lastPrice"]) if ticker.get("lastPrice") is not None else None
 
        if cg_price is None or binance_price is None:
            log(f"Missing price for {coin_id} - skipping.")
            continue
 
        spread_usd = binance_price - cg_price
        price_diff_pct = abs(spread_usd) / cg_price * 100
 
        high_price = float(ticker["highPrice"]) if ticker.get("highPrice") is not None else None
        low_price = float(ticker["lowPrice"]) if ticker.get("lowPrice") is not None else None
        binance_volume_usd = float(ticker["quoteVolume"]) if ticker.get("quoteVolume") is not None else None
        cg_volume_usd = cg_values.get("usd_24h_vol")
 
        daily_range_pct = None
        if high_price is not None and low_price is not None and binance_price:
            daily_range_pct = (high_price - low_price) / binance_price * 100
 
        binance_volume_share_pct = None
        if binance_volume_usd is not None and cg_volume_usd:
            binance_volume_share_pct = binance_volume_usd / cg_volume_usd * 100
 
        rows.append({
            "date": today,
            "coingecko_id": coin_id,
            "binance_id": binance_symbol,
            "cg_price": cg_price,
            "cg_volume_usd": cg_volume_usd,
            "market_cap_usd": cg_values.get("usd_market_cap"),
            "binance_price": binance_price,
            "open_price": float(ticker["openPrice"]) if ticker.get("openPrice") is not None else None,
            "high_price": high_price,
            "low_price": low_price,
            "binance_volume_usd": binance_volume_usd,
            "spread_usd": spread_usd,
            "price_diff_pct": price_diff_pct,
            "premium_exchange": "binance" if spread_usd > 0 else "coingecko",
            "is_arbitrage_viable": price_diff_pct > 0.35,
            "daily_range_pct": daily_range_pct,
            "binance_volume_share_pct": binance_volume_share_pct,
        })
 
    log(f"Parsed {len(rows)} coins with both CoinGecko and Binance data today.")
    return rows

In [14]:
def compute_rolling_for_coin(history, today_date, today_price):

    history_df = pd.DataFrame(history, columns=["date", "binance_price"])
    today_row = pd.DataFrame([{"date": today_date, "binance_price": today_price}])
    combined = pd.concat([history_df, today_row], ignore_index=True)
    combined = combined.drop_duplicates(subset=["date"], keep="last").sort_values("date").reset_index(drop=True)

    combined["binance_price"] = pd.to_numeric(combined["binance_price"], errors="coerce")
 
    combined["daily_return_pct"] = combined["binance_price"].pct_change() * 100
    combined["volatility_7d"] = combined["daily_return_pct"].rolling(window=7, min_periods=1).std()
 
    last = combined.iloc[-1]
    daily_return_pct = None if pd.isna(last["daily_return_pct"]) else float(last["daily_return_pct"])
    volatility_7d = 0.0 if pd.isna(last["volatility_7d"]) else float(last["volatility_7d"])
    return daily_return_pct, volatility_7d

In [10]:
def add_rolling_stats(conn, rows):
    cursor = conn.cursor()
    for row in rows:
        cursor.execute("""
            SELECT date, binance_price FROM daily_price_metrics
            WHERE coingecko_id = %s
            ORDER BY date DESC
            LIMIT 7;
        """, (row["coingecko_id"],))
        history = cursor.fetchall()
        daily_return_pct, volatility_7d = compute_rolling_for_coin(history, row["date"], row["binance_price"])
        row["daily_return_pct"] = daily_return_pct
        row["volatility_7d"] = volatility_7d
    cursor.close()
    return rows

In [11]:
def save_metrics(conn, rows):
    cursor = conn.cursor()
    upsert_query = """
        INSERT INTO daily_price_metrics (
            date, coingecko_id, binance_id, cg_price, cg_volume_usd, market_cap_usd,
            binance_price, open_price, high_price, low_price, binance_volume_usd,
            spread_usd, price_diff_pct, premium_exchange, daily_return_pct,
            volatility_7d, daily_range_pct, binance_volume_share_pct, is_arbitrage_viable
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT (date, coingecko_id)
        DO UPDATE SET
            cg_price = EXCLUDED.cg_price,
            cg_volume_usd = EXCLUDED.cg_volume_usd,
            market_cap_usd = EXCLUDED.market_cap_usd,
            binance_price = EXCLUDED.binance_price,
            open_price = EXCLUDED.open_price,
            high_price = EXCLUDED.high_price,
            low_price = EXCLUDED.low_price,
            binance_volume_usd = EXCLUDED.binance_volume_usd,
            spread_usd = EXCLUDED.spread_usd,
            price_diff_pct = EXCLUDED.price_diff_pct,
            premium_exchange = EXCLUDED.premium_exchange,
            daily_return_pct = EXCLUDED.daily_return_pct,
            volatility_7d = EXCLUDED.volatility_7d,
            daily_range_pct = EXCLUDED.daily_range_pct,
            binance_volume_share_pct = EXCLUDED.binance_volume_share_pct,
            is_arbitrage_viable = EXCLUDED.is_arbitrage_viable;
    """
    for row in rows:
        cursor.execute(upsert_query, (
            row["date"], row["coingecko_id"], row["binance_id"], row["cg_price"],
            row["cg_volume_usd"], row["market_cap_usd"], row["binance_price"],
            row["open_price"], row["high_price"], row["low_price"], row["binance_volume_usd"],
            row["spread_usd"], row["price_diff_pct"], row["premium_exchange"],
            row["daily_return_pct"], row["volatility_7d"], row["daily_range_pct"],
            row["binance_volume_share_pct"], row["is_arbitrage_viable"],
        ))
    conn.commit()
    cursor.close()
    log(f"Saved {len(rows)} rows to daily_price_metrics.")

In [15]:
def run():
    log("--- Starting daily pipeline run ---")
    load_dotenv()
    coingecko_api_key = os.environ["COINGECKO_API_KEY"]
    database_url = os.environ["DATABASE_URL"]
 
    coin_ids = list(COIN_TO_BINANCE.keys())
    binance_symbols = [s for s in COIN_TO_BINANCE.values() if s is not None]
    today = date.today()
 
    session = make_session()
    cg_status, cg_payload = fetch_coingecko(session, coingecko_api_key, coin_ids)
    bn_status, bn_payload = fetch_binance(session, binance_symbols)
 
    conn = psycopg2.connect(database_url)
 
    save_raw(conn, today, cg_status, cg_payload, bn_status, bn_payload)
 
    if cg_payload and bn_payload:
        rows = build_metrics_rows(today, cg_payload, bn_payload)
        rows = add_rolling_stats(conn, rows)
        save_metrics(conn, rows)
    else:
        log("Skipping daily_price_metrics update - one or both sources failed today.")
 
    conn.close()
    log("--- Daily pipeline run complete ---\n")

In [16]:
if __name__ == "__main__":
    run()

[2026-09-26T17:08:58.832353+00:00] --- Starting daily pipeline run ---
[2026-09-26T17:09:02.565440+00:00] Raw payloads saved to raw_ingestion_log.
[2026-09-26T17:09:02.568434+00:00] No Binance data for tether (None) today - skipping this coin.
[2026-09-26T17:09:02.570433+00:00] Parsed 19 coins with both CoinGecko and Binance data today.
[2026-09-26T17:09:14.534391+00:00] Saved 19 rows to daily_price_metrics.
[2026-09-26T17:09:14.538389+00:00] --- Daily pipeline run complete ---

